In [1]:
"""
SHAP attribution — 12 cities (Houston + Phoenix + 10 new)
- Train RF per city on A00-A63 (embed-only)
- SHAP TreeExplainer → shap_values CSV + summary plot + top dims table
"""

import pandas as pd, numpy as np, shap, matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.ensemble import RandomForestRegressor

ROOT = Path("..").resolve()  # notebooks/ is one level below project root
df   = pd.read_csv(ROOT / "outputs/modeling_table.csv", dtype={"GEOID": str})
EMBED_COLS = [f"A{i:02d}" for i in range(64)]
CITIES     = sorted(df["city"].unique().tolist())

def compute_gap_z(grp):
    def zs(s): return (s - s.mean()) / s.std() if s.std() > 0 else s * 0
    return zs(grp["hi_c"]) - zs(grp["lst_c"])

df["gap_z"] = df.groupby("city", group_keys=False).apply(
    compute_gap_z, include_groups=False
)

(ROOT / "outputs").mkdir(exist_ok=True)
all_top_dims = []

for city in CITIES:
    sub = df[df["city"] == city].dropna(subset=EMBED_COLS + ["gap_z"]).copy()
    X   = sub[EMBED_COLS]
    y   = sub["gap_z"]

    if len(sub) < 20:
        print(f"{city}: skipped ({len(sub)} rows)")
        continue

    rf = RandomForestRegressor(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X, y)
    train_r2 = rf.score(X, y)
    print(f"\n{city.upper()}  (n={len(sub)}, train R²={train_r2:.3f})")

    # SHAP
    explainer   = shap.TreeExplainer(rf)
    shap_values = explainer.shap_values(X)

    # Save full SHAP values
    shap_df = pd.DataFrame(shap_values, columns=EMBED_COLS)
    shap_df.insert(0, "GEOID", sub["GEOID"].values)
    shap_df.to_csv(ROOT / f"outputs/{city}_shap_values.csv", index=False)

    # Summary plot
    plt.figure(figsize=(10, 6))
    shap.summary_plot(shap_values, X, show=False)
    plt.title(f"{city.title()} — SHAP Summary (embedding dims)", fontsize=13)
    plt.tight_layout()
    plt.savefig(ROOT / f"outputs/{city}_shap_summary.png", dpi=200)
    plt.close()

    # Top 10 dims by mean |SHAP|
    importance = pd.DataFrame({
        "city":          city,
        "feature":       EMBED_COLS,
        "mean_abs_shap": np.mean(np.abs(shap_values), axis=0)
    }).sort_values("mean_abs_shap", ascending=False)
    print(importance.head(10).to_string(index=False))
    all_top_dims.append(importance.head(10))

# Cross-city top-dim summary
summary = pd.concat(all_top_dims, ignore_index=True)
summary.to_csv(ROOT / "outputs/all_cities_top_shap_dims.csv", index=False)
print("\nSaved:")
print("  outputs/<city>_shap_values.csv  (per city)")
print("  outputs/<city>_shap_summary.png (per city)")
print("  outputs/all_cities_top_shap_dims.csv")


/Users/chenchenmengmeng/Documents/Development/Projects/Sigspatial/rq1_explainability/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



ATLANTA  (n=223, train R²=0.971)
   city feature  mean_abs_shap
atlanta     A55       0.525194
atlanta     A00       0.324821
atlanta     A56       0.204158
atlanta     A10       0.137862
atlanta     A29       0.072954
atlanta     A61       0.042276
atlanta     A49       0.041821
atlanta     A48       0.035284
atlanta     A38       0.030949
atlanta     A57       0.027332

BOSTON  (n=191, train R²=0.924)
  city feature  mean_abs_shap
boston     A16       0.229497
boston     A15       0.204478
boston     A19       0.080417
boston     A32       0.077085
boston     A46       0.067940
boston     A41       0.056546
boston     A26       0.053093
boston     A09       0.052735
boston     A03       0.051856
boston     A33       0.050405

CHICAGO  (n=872, train R²=0.935)
   city feature  mean_abs_shap
chicago     A00       0.127259
chicago     A01       0.089093
chicago     A57       0.070075
chicago     A09       0.067116
chicago     A18       0.051156
chicago     A30       0.037377
chicago    